<a href="https://colab.research.google.com/github/xD0ri4nx/bot_investitiV2/blob/main/bot_invetitii_collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install streamlit yfinance scikit-learn tensorflow pandas numpy google-genai

In [8]:
import pandas as pd
import time
import os
import json
from datetime import datetime
from google import genai
from google.genai import types
from google.colab import userdata

# ---- 1. SECURE AGENT CONFIGURATION ----
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=API_KEY)
except userdata.SecretNotFoundError:
    print("[CRITICAL ERROR] Secret 'GEMINI_API_KEY' not found.")
    raise

# NOU: Array of all the assets in your Streamlit App
TICKERS = ["AAPL", "NVDA", "MSFT", "SPY", "BTC-USD", "ETH-USD"]

sentiment_schema = {
    "type": "object",
    "properties": {
        "sentiment_score": {"type": "number", "description": "Float from -1.0 to 1.0."}
    },
    "required": ["sentiment_score"]
}

# ---- 2. REASONING ENGINE (GEMINI) ----
# NOU: The function now takes the specific 'ticker' as an argument
def analyze_daily_news(ticker, date, headlines_list):
    if not headlines_list:
        return 0.0

    daily_news_text = "\n".join([f"- {h}" for h in headlines_list])

    prompt = f"""
    You are an elite quantitative financial analyst for a hedge fund.
    Review the following news headlines for {ticker} published on {date}.

    Calculate an aggregate daily sentiment score from -1.0 to 1.0.
    Ignore general market noise; focus strictly on how this news affects {ticker}.

    Headlines:
    {daily_news_text}
    """

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1,
                response_mime_type="application/json",
                response_schema=sentiment_schema,
            ),
        )
        result = json.loads(response.text)
        return result.get("sentiment_score", 0.0)
    except Exception as e:
        print(f"[!] API Error for {ticker} on date {date}: {e}")
        return 0.0


# ---- 3. DATA SIMULATOR ----
# NOU: The data fetcher now accepts a ticker so you can load the specific CSV for that asset
def fetch_raw_historical_news(ticker):
    print(f"Extracting raw news database for {ticker}...")

    # In a real scenario, you would do:
    # df = pd.read_csv(f'{ticker}_raw_news.csv')

    # For demonstration, we use generic mock data
    mock_news_database = {
        "2024-02-21": [f"{ticker} announces historic quarterly earnings", f"Wall Street upgrades {ticker}"],
        "2022-09-01": [f"Macro conditions threaten {ticker} supply chain", f"Investors sell off {ticker} stock"]
    }
    return mock_news_database


# ---- 4. MULTI-ASSET BATCH PIPELINE ----
def run_sentiment_miner():
    print("--- STARTING MULTI-ASSET NLP PIPELINE ---")

    # Loop through EVERY ticker in your list
    for ticker in TICKERS:
        print(f"\n================ Processing {ticker} ================")
        historical_news = fetch_raw_historical_news(ticker)

        results = []
        total_days = len(historical_news)

        for i, (date, headlines) in enumerate(historical_news.items()):
            print(f"[{ticker}] Processing day {i+1}/{total_days} [{date}]... ", end="")

            score = analyze_daily_news(ticker, date, headlines)
            results.append({"Date": date, "Sentiment_Score": score})

            print(f"Score: {score}")
            time.sleep(1.5) # Rate limiting

        # Save a unique CSV for THIS specific ticker
        df_results = pd.DataFrame(results)
        df_results['Date'] = pd.to_datetime(df_results['Date'])
        df_results.sort_values('Date', inplace=True)

        output_filename = f"{ticker}_sentiment.csv"
        df_results.to_csv(output_filename, index=False)
        print(f"[SUCCESS] Saved {output_filename}")

# Execute the pipeline
run_sentiment_miner()

--- STARTING MULTI-ASSET NLP PIPELINE ---

================ Processing AAPL ================
Extracting raw news database for AAPL...
[AAPL] Processing day 1/2 [2024-02-21]... Score: 0.95
[AAPL] Processing day 2/2 [2022-09-01]... Score: -0.9
[SUCCESS] Saved AAPL_sentiment.csv

================ Processing NVDA ================
Extracting raw news database for NVDA...
[NVDA] Processing day 1/2 [2024-02-21]... Score: 0.95
[NVDA] Processing day 2/2 [2022-09-01]... Score: -0.9
[SUCCESS] Saved NVDA_sentiment.csv

================ Processing MSFT ================
Extracting raw news database for MSFT...
[MSFT] Processing day 1/2 [2024-02-21]... Score: 0.95
[MSFT] Processing day 2/2 [2022-09-01]... Score: -0.9
[SUCCESS] Saved MSFT_sentiment.csv

================ Processing SPY ================
Extracting raw news database for SPY...
[SPY] Processing day 1/2 [2024-02-21]... [!] API Error for SPY on date 2024-02-21: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu

In [9]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, Input, Bidirectional, Conv1D, GRU,
    BatchNormalization, GaussianNoise, LayerNormalization, Add
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
from tensorflow.keras.regularizers import l2
import tensorflow as tf
import warnings
import os
import gc

# ---- 1. SYSTEM CONFIGURATION & T4 GPU OPTIMIZATION ----
tf.random.set_seed(42)

# Enable Mixed Precision (FP16) - Unleashes NVIDIA T4 Tensor Cores
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

st.set_page_config(page_title="Institutional Quant AI", layout="wide", initial_sidebar_state="expanded")

# ---- CUSTOM CSS (GLASSMORPHISM) ----
st.markdown("""
<style>
    .stApp { background: linear-gradient(135deg, #0b132b, #1c2541, #3a506b); color: #ffffff; }
    header[data-testid="stHeader"] { background: transparent !important; }
    [data-testid="stSidebar"] {
        background: rgba(28, 37, 65, 0.4) !important;
        backdrop-filter: blur(15px);
        -webkit-backdrop-filter: blur(15px);
        border-right: 1px solid rgba(255, 255, 255, 0.1);
    }
    [data-testid="stMetric"] {
        background: rgba(255, 255, 255, 0.05); backdrop-filter: blur(12px);
        -webkit-backdrop-filter: blur(12px); border: 1px solid rgba(255, 255, 255, 0.1);
        border-radius: 12px; padding: 20px; box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.3);
        transition: transform 0.2s ease, box-shadow 0.2s ease;
    }
    [data-testid="stMetric"]:hover {
        transform: translateY(-2px); box-shadow: 0 12px 40px 0 rgba(0, 0, 0, 0.4);
        background: rgba(255, 255, 255, 0.08);
    }
    div.stButton > button {
        background: linear-gradient(90deg, #3a506b, #1c2541); color: white;
        border: 1px solid rgba(255, 255, 255, 0.2); border-radius: 8px;
        padding: 10px 24px; font-weight: 600; letter-spacing: 0.5px;
        transition: all 0.3s ease; box-shadow: 0 4px 15px rgba(0,0,0,0.2); width: 100%;
    }
    div.stButton > button:hover {
        background: linear-gradient(90deg, #5bc0be, #3a506b);
        border: 1px solid rgba(255, 255, 255, 0.5);
        box-shadow: 0 0 20px rgba(91, 192, 190, 0.4); transform: translateY(-1px);
    }
    [data-testid="stStatusWidget"] {
        background: rgba(0, 0, 0, 0.2) !important; backdrop-filter: blur(10px);
        border: 1px solid rgba(255, 255, 255, 0.1); border-radius: 10px;
    }
    .streamlit-expanderHeader { background: rgba(255, 255, 255, 0.05); border-radius: 8px; }
    h1, h2, h3, p, span, div, label { color: #e0e0e0; }
    [data-testid="stMetricValue"] { color: #ffffff !important; }

    *:focus, *:focus-visible, *:focus-within { outline: none !important; }
    .stSelectbox [data-baseweb="select"], .stDateInput [data-baseweb="input"] { background-color: transparent !important; }
    div[data-baseweb="select"] > div, div[data-baseweb="input"] > div {
        background: linear-gradient(90deg, #1c2541, #3a506b) !important;
        border: 1px solid rgba(255, 255, 255, 0.2) !important; border-radius: 8px !important;
        box-shadow: none !important; transition: all 0.3s ease !important;
    }
    div[data-baseweb="select"] > div:focus-within, div[data-baseweb="input"] > div:focus-within {
        border-color: #5bc0be !important; box-shadow: 0 0 0 1px #5bc0be !important;
    }
    div[data-baseweb="select"] > div:hover, div[data-baseweb="input"] > div:hover {
        background: linear-gradient(90deg, #2a3a5a, #4a6a8a) !important;
        border: 1px solid rgba(91, 192, 190, 0.5) !important;
    }
    div[data-baseweb="select"] *, div[data-baseweb="input"] input { color: white !important; background: transparent !important; }
</style>
""", unsafe_allow_html=True)

st.title("Institutional Quant AI: Sniper Alpha Edition")

st.markdown("""
### System Architecture (Thesis Final)
* **Apex Engine:** WaveNet-style Causal TCN $\\rightarrow$ L2 Regularized Bi-GRU (Anti-Overfit).
* **Execution Logic:** Strict **Conviction Threshold** to act as a sniper, drastically reducing transaction fees.
* **T4 Optimization:** FP16 Tensor Cores + `tf.data.Dataset` pipelines.
""")

# ---- 2. QUANT PIPELINE ----
class QuantPipeline:
    def __init__(self, symbol, start_date, end_date, time_step):
        self.symbol = symbol
        self.start_date = start_date
        self.end_date = end_date
        self.time_step = time_step
        self.scaler = RobustScaler()

    def fetch_data(self):
        try:
            df = yf.download(self.symbol, start=self.start_date, end=self.end_date, progress=False)
            if len(df) == 0: raise ValueError("No data found.")

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            req_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
            if not all(col in df.columns for col in req_cols):
                 if 'Adj Close' in df.columns:
                     df['Close'] = df['Adj Close']

            return df
        except Exception as e:
            st.error(f"Data Feed Error: {e}")
            st.stop()

    def engineer_features(self, df):
        df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
        df['Target'] = df['Log_Returns'].shift(-1)

        log_hl = np.log(df['High'] / df['Low'])**2
        log_co = np.log(df['Close'] / df['Open'])**2
        df['Garman_Klass'] = 0.5 * log_hl - (2 * np.log(2) - 1) * log_co

        delta = df['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        df['RSI'] = 100 - (100 / (1 + rs))

        high_low = df['High'] - df['Low']
        high_low = high_low.replace(0, 0.0001)
        ad_val = (((df['Close'] - df['Low']) - (df['High'] - df['Close'])) / high_low) * df['Volume']
        df['CMF'] = ad_val.rolling(20).sum() / df['Volume'].rolling(20).sum()
        df['CMF'] = df['CMF'].fillna(0).replace([np.inf, -np.inf], 0)

        ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
        ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
        df['MACD'] = ema_12 - ema_26
        df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
        df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

        sma_20 = df['Close'].rolling(window=20).mean()
        std_20 = df['Close'].rolling(window=20).std()
        bb_high = sma_20 + (std_20 * 2)
        bb_low = sma_20 - (std_20 * 2)
        df['BB_Position'] = (df['Close'] - bb_low) / (bb_high - bb_low).replace(0, 0.0001)

        df['TR'] = np.maximum(df['High'] - df['Low'],
                              np.maximum(abs(df['High'] - df['Close'].shift(1)),
                                         abs(df['Low'] - df['Close'].shift(1))))
        df['ATR'] = df['TR'].rolling(14).mean() / df['Close']

        df['MOM_5'] = df['Close'].pct_change(5)
        df['MOM_20'] = df['Close'].pct_change(20)

        t = df.index
        df['Day_Sin'] = np.sin(2 * np.pi * t.dayofweek / 7)
        df['Day_Cos'] = np.cos(2 * np.pi * t.dayofweek / 7)

        df.dropna(inplace=True)
        return df

    def prepare_tensors(self, df, split_ratio=0.8):
        feature_cols = ['Log_Returns', 'Garman_Klass', 'RSI', 'CMF', 'MACD_Hist',
                        'BB_Position', 'ATR', 'MOM_5', 'MOM_20', 'Day_Sin', 'Day_Cos']

        split_idx = int(len(df) * split_ratio)
        train_df = df.iloc[:split_idx]
        test_df = df.iloc[split_idx:]

        decay_rate = 0.001
        dates = np.arange(len(train_df))
        weights = np.exp(decay_rate * dates)
        weights = weights / weights.mean()

        X_train_scaled = self.scaler.fit_transform(train_df[feature_cols])
        X_test_scaled = self.scaler.transform(test_df[feature_cols])

        y_train = train_df['Target'].values
        y_test = test_df['Target'].values

        def create_window(X, y, time_step, w=None):
            Xs, ys, ws = [], [], []
            for i in range(len(X) - time_step):
                Xs.append(X[i:(i + time_step)])
                ys.append(y[i + time_step])
                if w is not None: ws.append(w[i + time_step])
            if w is not None:
                return np.array(Xs), np.array(ys), np.array(ws)
            return np.array(Xs), np.array(ys)

        X_train_3d, y_train_3d, w_train = create_window(X_train_scaled, y_train, self.time_step, weights)
        X_test_3d, y_test_3d = create_window(X_test_scaled, y_test, self.time_step)

        return X_train_3d, y_train_3d, w_train, X_test_3d, y_test_3d, test_df.iloc[self.time_step:]

    def build_model(self, input_shape):
        tf.keras.backend.clear_session()
        gc.collect()

        inputs = Input(shape=input_shape)
        x = GaussianNoise(0.01)(inputs)
        x = BatchNormalization()(x)

        # 1. State-of-the-Art TCN Block (Temporal Convolutional Network)
        # Using CAUSAL padding ensures zero future-data leakage. Dilation captures long trends.
        conv1 = Conv1D(filters=64, kernel_size=3, dilation_rate=1, padding='causal', activation='gelu')(x)
        conv2 = Conv1D(filters=64, kernel_size=3, dilation_rate=2, padding='causal', activation='gelu')(conv1)
        conv3 = Conv1D(filters=64, kernel_size=3, dilation_rate=4, padding='causal', activation='gelu')(conv2)

        # Residual skip connection to prevent gradient degradation
        x = Add()([conv1, conv3])
        x = LayerNormalization()(x)

        # 2. Bi-GRU (Gated Recurrent Unit)
        # GRUs are vastly superior to LSTMs for noisy financial data because they have fewer parameters, stopping overfitting.
        x = Bidirectional(GRU(128, return_sequences=False, kernel_regularizer=l2(1e-4)))(x)
        x = Dropout(0.4)(x)

        # 3. Dense Head
        x = Dense(64, activation='gelu', kernel_regularizer=l2(1e-4))(x)
        x = Dropout(0.3)(x)

        outputs = Dense(1, activation='linear', dtype='float32')(x)

        model = Model(inputs=inputs, outputs=outputs)
        optimizer = Adam(learning_rate=0.0005)
        model.compile(optimizer=optimizer, loss='huber')

        return model


# ---- 3. UI & PRESET CONFIGURATIONS ----
IDEAL_CONFIGS = {
    "AAPL": {"start": "2018-01-01", "long_only": True, "cost_bps": 10, "target_vol": 30, "threshold": 0.20},
    "NVDA": {"start": "2018-01-01", "long_only": True, "cost_bps": 10, "target_vol": 60, "threshold": 0.30},
    "MSFT": {"start": "2018-01-01", "long_only": True, "cost_bps": 10, "target_vol": 30, "threshold": 0.20},
    "SPY": {"start": "2010-01-01", "long_only": True, "cost_bps": 5, "target_vol": 15, "threshold": 0.10},
    "BTC-USD": {"start": "2019-01-01", "long_only": False, "cost_bps": 20, "target_vol": 80, "threshold": 0.40},
    "ETH-USD": {"start": "2019-01-01", "long_only": False, "cost_bps": 20, "target_vol": 90, "threshold": 0.40},
}

with st.sidebar:
    st.header("Strategy Config")

    ticker = st.selectbox("Asset Class", list(IDEAL_CONFIGS.keys()))
    cfg = IDEAL_CONFIGS[ticker]

    start = st.date_input("Start", value=pd.to_datetime(cfg["start"]))
    end = st.date_input("End", value=pd.to_datetime("today"))

    st.subheader("Risk Management")
    long_only = st.checkbox("Long Only Mode (No Shorts)", value=cfg["long_only"])

    ensemble_size = st.slider("Ensemble Size", 1, 10, 5)
    cost_bps = st.slider("Transaction Cost (bps)", 0, 50, cfg["cost_bps"])
    target_vol = st.slider("Target Volatility (%)", 10, 100, cfg["target_vol"])

    # THE PROFIT SAVER: Conviction Threshold
    conviction_thresh = st.slider("Conviction Deadband", 0.0, 1.0, cfg["threshold"],
                                  help="If AI conviction is below this, it holds cash. Saves massive fees.")

if st.button("Launch Strategy"):
    pipeline = QuantPipeline(ticker, start, end, 60)

    with st.status("Initializing Quantum Engine...", expanded=True) as status:

        st.write("Fetching institutional data feed...")
        raw_df = pipeline.fetch_data()

        st.write("Calculating Advanced Alpha Features...")
        processed_df = pipeline.engineer_features(raw_df)

        st.write("Applying Exponential Decay Weights...")
        X_train, y_train, w_train, X_test, y_test, test_context = pipeline.prepare_tensors(processed_df)

        st.write("Constructing high-throughput GPU data pipelines...")
        BATCH_SIZE = 128

        train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train, w_train))
        train_dataset = train_dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
        test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        st.write(f"Training Sniper Ensemble of {ensemble_size} TCN-GRU Models...")
        ensemble_preds = []
        progress_bar = st.progress(0)

        for i in range(ensemble_size):
            tf.random.set_seed(42 + i)
            st.write(f"Training Neural Network {i+1}/{ensemble_size} on GPU Tensor Cores")

            model = pipeline.build_model((X_train.shape[1], X_train.shape[2]))

            early_stop = EarlyStopping(monitor='val_loss', patience=18, restore_best_weights=True)
            reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=6, min_lr=1e-6)

            model.fit(
                train_dataset,
                validation_data=test_dataset,
                epochs=60,
                callbacks=[early_stop, reduce_lr],
                verbose=0
            )

            preds = model.predict(test_dataset, verbose=0).flatten()
            ensemble_preds.append(preds)
            progress_bar.progress((i + 1) / ensemble_size)

        st.write("Aggregating predictions & Applying Conviction Deadband...")
        avg_preds = np.mean(ensemble_preds, axis=0)

        status.update(label="Optimization Complete", state="complete", expanded=False)

    # ---- 4. BACKTEST ENGINE ----
    backtest_df = test_context.copy()
    backtest_df['Predicted_Return'] = avg_preds
    backtest_df['Actual_Return'] = y_test

    backtest_df['SMA_50'] = backtest_df['Close'].rolling(window=50).mean()
    backtest_df['Trend_Bullish'] = backtest_df['Close'] > backtest_df['SMA_50']

    current_vol = np.sqrt(backtest_df['Garman_Klass']) * np.sqrt(252)
    safe_vol = (current_vol * 100).replace(0, 0.01)
    backtest_df['Vol_Scalar'] = target_vol / safe_vol
    backtest_df['Vol_Scalar'] = backtest_df['Vol_Scalar'].clip(upper=2.0)

    boost_factor = np.where(backtest_df['Trend_Bullish'], 2.0, 1.0)

    # 1. Calculate Raw Conviction
    backtest_df['Conviction'] = tf.math.sigmoid(backtest_df['Predicted_Return'] * 500 * boost_factor).numpy() - 0.5
    backtest_df['Conviction'] = backtest_df['Conviction'] * 2

    # 2. APPLY THE DEADBAND (The Sniper Logic)
    # If the conviction is less than our threshold, set it to exactly 0 (Hold Cash)
    backtest_df['Conviction'] = np.where(backtest_df['Conviction'].abs() >= conviction_thresh, backtest_df['Conviction'], 0.0)

    raw_position = backtest_df['Conviction'] * backtest_df['Vol_Scalar']

    if long_only:
        backtest_df['Position'] = raw_position.clip(lower=0)
    else:
        backtest_df['Position'] = raw_position

    backtest_df['Position'] = backtest_df['Position'].fillna(0)

    # Cost & PnL
    backtest_df['Turnover'] = backtest_df['Position'].diff().abs().fillna(0)
    backtest_df['Cost'] = backtest_df['Turnover'] * (cost_bps / 10000)

    backtest_df['Gross_Return'] = backtest_df['Position'] * backtest_df['Actual_Return']
    backtest_df['Net_Return'] = backtest_df['Gross_Return'] - backtest_df['Cost']

    backtest_df['Cum_Market'] = (1 + backtest_df['Actual_Return']).cumprod()
    backtest_df['Cum_Strategy'] = (1 + backtest_df['Net_Return']).cumprod()

    # Metrics
    total_ret = backtest_df['Cum_Strategy'].iloc[-1] - 1
    sharpe = np.sqrt(252) * (backtest_df['Net_Return'].mean() / backtest_df['Net_Return'].std())
    peak = backtest_df['Cum_Strategy'].cummax()
    max_dd = ((backtest_df['Cum_Strategy'] - peak) / peak).min()

    wins = backtest_df[backtest_df['Net_Return'] > 0]['Net_Return']

    # THE FIX: Calculate Realized Correctness BEFORE slicing for active trading days
    backtest_df['Realized_Correctness'] = np.sign(backtest_df['Predicted_Return']) == np.sign(backtest_df['Net_Return'])

    # Only calculate win rate on days the bot actually traded (Conviction > 0)
    active_trading_days = backtest_df[backtest_df['Position'].abs() > 0]

    win_rate = len(wins) / len(active_trading_days) if len(active_trading_days) > 0 else 0
    accuracy = active_trading_days['Realized_Correctness'].mean() * 100 if len(active_trading_days) > 0 else 0

    st.divider()

    st.container()
    c1, c2, c3, c4, c5 = st.columns(5)
    c1.metric("Net Profit", f"{total_ret*100:.2f}%")
    c2.metric("Sharpe Ratio", f"{sharpe:.2f}")
    c3.metric("Win Rate", f"{win_rate*100:.1f}%")
    c4.metric("Max Drawdown", f"{max_dd*100:.2f}%")
    c5.metric("Realized Accuracy", f"{accuracy:.1f}%")

    st.markdown("<br>", unsafe_allow_html=True)
    st.subheader("Performance (Net of Fees)")
    st.line_chart(backtest_df[['Cum_Strategy', 'Cum_Market']])

    with st.expander("See Trade Log"):
        st.dataframe(backtest_df[['Predicted_Return', 'Conviction', 'Position', 'Cost', 'Net_Return']].tail(100))
        csv = backtest_df.to_csv().encode('utf-8')
        st.download_button("Download Data", csv, "backtest_data.csv", "text/csv")

Overwriting app.py


In [1]:
import subprocess
import time
import os
from google.colab import output
from google.colab.output import eval_js

print("Cleaning up old background processes...")
os.system("pkill -f streamlit")
time.sleep(1)

print("Starting Streamlit server...")
# 1. We removed the DEVNULL silencers so we can see if it crashes
# 2. We added back the 0.0.0.0 binding and CORS bypass
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.headless", "true",
    "--server.address", "0.0.0.0",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

# Give the server 5 full seconds to initialize before the proxy connects
time.sleep(5)

print("Opening native Colab proxy...")
proxy_url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"Click here for full-screen dashboard: {proxy_url}")

print("Loading embedded dashboard...")
output.serve_kernel_port_as_iframe(8501, height=720)

Cleaning up old background processes...
Starting Streamlit server...


FileNotFoundError: [Errno 2] No such file or directory: 'streamlit'